In [45]:
# Importing necessary libraries
import numpy as np
import pandas as pd
import matplotlib, os, datetime, sys, os, re
import matplotlib.style as style 
import matplotlib.pyplot as plt
import seaborn as sns

In [46]:
import importlib
%matplotlib inline
#sns.set(style="ticks")
# Making sure that all columns are displyed when we look at data
pd.set_option('display.max_columns', None) 
style.use('ggplot')
#style.use('seaborn') This didn't seem to work 
sns.set(font_scale=1.2)

In [47]:
year = 2050

In [48]:
# dataFolder = 'data'
# configFolder = 'configs'
# runFolder = 'output'
# runFolder_GQ = 'output_gq' 

# Merging HH and GQ populations

## Reading syntheic data

In [49]:
# reading GQ syn hh as it has Type
# hhgq_fname = os.path.join(runFolder_GQ, 'synthetic_households_gq.csv')
# hh_fname = os.path.join(runFolder, 'synthetic_households.csv')
hhgq_df = pd.read_csv(fr'T:\socioec\population sim\3. Post_processing\{year}\synthetic_households_gq.csv')
hh_df = pd.read_csv(fr'T:\socioec\population sim\3. Post_processing\{year}\synthetic_households.csv')
print('hh_df shape: ', hh_df.shape)
print('hhgq_df shape: ', hhgq_df.shape)

hh_df shape:  (1334543, 10)
hhgq_df shape:  (114574, 11)


In [50]:
# reading GQ syn hh as it has Type
# pergq_fname = os.path.join(runFolder_GQ, 'synthetic_persons_gq.csv')
# per_fname = os.path.join(runFolder, 'synthetic_persons.csv')
pergq_df = pd.read_csv(fr'T:\socioec\population sim\3. Post_processing\{year}\synthetic_persons_gq.csv')
per_df = pd.read_csv(fr'T:\socioec\population sim\3. Post_processing\{year}\synthetic_persons.csv')

print('per_df shape: ', per_df.shape)
print('pergq_df shape: ', pergq_df.shape)

# correcting the household id in gq for it to be unique
hhgq_df['household_id'] = len(hh_df) + hhgq_df['household_id']
pergq_df['household_id'] = len(hh_df) + pergq_df['household_id']


per_df shape:  (3283012, 18)
pergq_df shape:  (114574, 18)


In [51]:
hhgq_df.head()
set(hhgq_df['GQ_type'])

{1, 2, 3}

In [52]:
hh_df.head()

,household_id,PUMA,mgra,NP,HHADJINC,HHT,WIF,HUPAC,VEH,BLD
0,1,7301,57,2,196767.9,1.0,2.0,4.0,2.0,2.0
1,2,7301,57,3,559393.8,1.0,2.0,4.0,3.0,2.0
2,3,7301,57,1,44264.7,6.0,NaN,4.0,1.0,3.0
3,4,7301,57,4,12924.0,3.0,1.0,1.0,2.0,3.0
4,5,7301,57,3,141087.0,2.0,1.0,1.0,2.0,3.0


In [53]:
# Adding tract to HH GQ and PER GQ
# hhgq_df = pd.merge(hhgq_df, geodf[['mgra']], 
#                                on = 'mgra', how = 'left')
# pergq_df = pd.merge(pergq_df, geodf[['mgra']], 
#                                on = 'mgra', how = 'left')

In [54]:
hhfull_df = pd.concat([hh_df, hhgq_df])
hhfull_df = hhfull_df[[item for item in hhfull_df.columns if item not in ['PUMA']]]
print("hhfull_df shape: ", hhfull_df.shape)
hhfull_df.head()

hhfull_df shape:  (1449117, 10)


,household_id,mgra,NP,HHADJINC,HHT,WIF,HUPAC,VEH,BLD,GQ_type
0,1,57,2,196767.9,1.0,2.0,4.0,2.0,2.0,NaN
1,2,57,3,559393.8,1.0,2.0,4.0,3.0,2.0,NaN
2,3,57,1,44264.7,6.0,NaN,4.0,1.0,3.0,NaN
3,4,57,4,12924.0,3.0,1.0,1.0,2.0,3.0,NaN
4,5,57,3,141087.0,2.0,1.0,1.0,2.0,3.0,NaN


In [55]:
hhfull_df['HHADJINC'] = hhfull_df['HHADJINC'].apply(lambda x : x if x > 0 else 0)

In [56]:
perfull_df = pd.concat([per_df, pergq_df])
perfull_df = perfull_df[[item for item in perfull_df.columns if item not in ['PUMA']]]
print("perfull_df shape: ", perfull_df.shape)
perfull_df.head()

perfull_df shape:  (3397586, 17)


,mgra,household_id,SPORDER,AGEP,SEX,ESR,COW,WKHP,SCHG,RAC1P,HISP,MIL,SCHL,OCCP,WKW,NAICS2,SOC2
0,57,1,1,57,1,1.0,1.0,50.0,0,1,1,4.0,21.0,2810.0,1.0,51,27
1,57,1,2,55,2,1.0,2.0,45.0,0,1,1,4.0,24.0,2205.0,1.0,61,25
2,57,2,1,68,1,1.0,4.0,32.0,0,9,1,4.0,23.0,3220.0,1.0,62,29
3,57,2,2,56,2,6.0,NaN,NaN,0,9,1,4.0,18.0,NaN,NaN,0,0
4,57,2,3,32,2,1.0,1.0,36.0,0,9,1,2.0,21.0,4621.0,1.0,62,39


In [57]:
temp_per = pd.merge(perfull_df, hhfull_df[['household_id', 'GQ_type']], on='household_id' )
temp_per['one'] = 1
temp_per['job_cat'] = None

In [58]:
temp_per

,mgra,household_id,SPORDER,AGEP,SEX,ESR,COW,WKHP,SCHG,RAC1P,HISP,MIL,SCHL,OCCP,WKW,NAICS2,SOC2,GQ_type,one,job_cat
0,57,1,1,57,1,1.0,1.0,50.0,0,1,1,4.0,21.0,2810.0,1.0,51,27,NaN,1,None
1,57,1,2,55,2,1.0,2.0,45.0,0,1,1,4.0,24.0,2205.0,1.0,61,25,NaN,1,None
2,57,2,1,68,1,1.0,4.0,32.0,0,9,1,4.0,23.0,3220.0,1.0,62,29,NaN,1,None
3,57,2,2,56,2,6.0,NaN,NaN,0,9,1,4.0,18.0,NaN,NaN,0,0,NaN,1,None
4,57,2,3,32,2,1.0,1.0,36.0,0,9,1,2.0,21.0,4621.0,1.0,62,39,NaN,1,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3397581,24280,1449113,1,65,1,6.0,6.0,40.0,0,1,1,4.0,18.0,440.0,1.0,81,11,3.0,1,None
3397582,24280,1449114,1,64,1,6.0,NaN,NaN,0,1,1,4.0,1.0,NaN,NaN,0,0,3.0,1,None
3397583,24280,1449115,1,29,1,6.0,1.0,NaN,0,2,1,4.0,16.0,9645.0,NaN,45,53,3.0,1,None
3397584,24280,1449116,1,49,1,6.0,NaN,NaN,0,1,24,4.0,17.0,NaN,NaN,0,0,3.0,1,None


In [59]:
# CRA added this
con_set_df1 = pd.read_csv(r'T:\socioec\population sim\2. Implementation\configs\controls.csv')
con_set_df1 = con_set_df1[con_set_df1['target'].str.contains('job')]
con_set_df1

,target,geography,seed_table,importance,control_field,expression
38,job_1,region,persons,350000,job_1,(persons.NAICS2 == '92')
39,job_10,region,persons,350000,job_10,(persons.NAICS2 == '22') | (persons.NAICS2 == ...
40,job_2,region,persons,350000,job_2,(persons.NAICS2 == 'MIL')
41,job_3,region,persons,350000,job_3,(persons.NAICS2 == '11') | (persons.NAICS2 == ...
42,job_4,region,persons,350000,job_4,(persons.NAICS2 == '51') | (persons.NAICS2 == ...
43,job_5,region,persons,350000,job_5,(persons.NAICS2 == '52') | (persons.NAICS2 == ...
44,job_6,region,persons,350000,job_6,(persons.NAICS2 == '61')
45,job_7,region,persons,350000,job_7,(persons.NAICS2 == '62')
46,job_8,region,persons,350000,job_8,(persons.NAICS2 == '44') | (persons.NAICS2 == ...
47,job_9,region,persons,350000,job_9,(persons.NAICS2 == '23') | (persons.NAICS2 == ...


In [60]:
for i, row in con_set_df1.iterrows():
    j, expr = row['target'], f"{row['expression']}"
    expr = expr.replace('persons', 'temp_per')
    temp_per.loc[pd.eval(expr), 'job_cat'] = j

print("temp_per shape: ", temp_per.shape)
temp_per.head()

temp_per shape:  (3397586, 20)


,mgra,household_id,SPORDER,AGEP,SEX,ESR,COW,WKHP,SCHG,RAC1P,HISP,MIL,SCHL,OCCP,WKW,NAICS2,SOC2,GQ_type,one,job_cat
0,57,1,1,57,1,1.0,1.0,50.0,0,1,1,4.0,21.0,2810.0,1.0,51,27,NaN,1,job_4
1,57,1,2,55,2,1.0,2.0,45.0,0,1,1,4.0,24.0,2205.0,1.0,61,25,NaN,1,job_6
2,57,2,1,68,1,1.0,4.0,32.0,0,9,1,4.0,23.0,3220.0,1.0,62,29,NaN,1,job_7
3,57,2,2,56,2,6.0,NaN,NaN,0,9,1,4.0,18.0,NaN,NaN,0,0,NaN,1,None
4,57,2,3,32,2,1.0,1.0,36.0,0,9,1,2.0,21.0,4621.0,1.0,62,39,NaN,1,job_7


In [61]:
# temp_per = pd.merge(perfull_df, hhfull_df[['household_id', 'GQ_type']], on='household_id' )
# temp_per['one'] = 1
# temp_per['job_cat'] = None
# for i, row in con_set_df1.iterrows():
#     j, expr = row['target'], f"{row['expression']}"
#     expr = expr.replace('persons', 'temp_per')
#     temp_per.loc[pd.eval(expr), 'job_cat'] = j

# print("temp_per shape: ", temp_per.shape)
# temp_per.head()

# Fixing null values 

In [62]:
#na cols
colsnahh = ['HHT', 'HUPAC', "BLD", 'GQ_type']
for col in colsnahh:
    hhfull_df[col] = hhfull_df[col].fillna(0)
hhfull_df.isna().sum()

household_id         0
mgra                 0
NP                   0
HHADJINC             0
HHT                  0
WIF             555190
HUPAC                0
VEH             114574
BLD                  0
GQ_type              0
dtype: int64

In [63]:
#perfull_df[ESR,COW, WKHP, SCHG, MIL, SCHL, OCCP, WKW]
colsnaper = ['ESR', 'COW', 'WKHP', 'SCHG', 'MIL', 'SCHL', 'OCCP', 'WKW']
for col in colsnaper:
    perfull_df[col] = perfull_df[col].fillna(0)
perfull_df.isna().sum()

mgra            0
household_id    0
SPORDER         0
AGEP            0
SEX             0
ESR             0
COW             0
WKHP            0
SCHG            0
RAC1P           0
HISP            0
MIL             0
SCHL            0
OCCP            0
WKW             0
NAICS2          0
SOC2            0
dtype: int64

In [64]:
hhfull_df['GQ_type'].value_counts()

GQ_type
0.0    1334543
3.0      42890
1.0      41181
2.0      30503
Name: count, dtype: int64

## Writing out full populations
-- save this in the T drive for ABM run

In [65]:
hhfull_df.to_csv(fr'T:\socioec\Current_Projects\SR15\S0\version7\abm_csv\synthetic_households_{year}_01.csv', index=False)
perfull_df.to_csv(fr'T:\socioec\Current_Projects\SR15\S0\version7\abm_csv\synthetic_persons_{year}_01.csv', index=False)

In [66]:
# hhfull_df.to_csv(os.path.join(analysisFolder, 'synthetic_households.csv'), index=False)
# perfull_df.to_csv(os.path.join(analysisFolder, 'synthetic_persons.csv'), index=False)